<div style="background: linear-gradient(135deg,#240A44 0%,#4A148C 100%);
            padding:40px 36px 32px 36px;
            border-radius:12px;
            color:white;
            font-family:Arial,sans-serif">

<div style="font-size:13px; letter-spacing:2px; opacity:.75; margin-bottom:8px">
UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026
</div>

<h1 style="margin:0 0 6px 0; font-size:28px; font-weight:700; line-height:1.2">
Modelado — Hurdle Model (dos etapas)
</h1>

<h2 style="margin:0 0 20px 0; font-size:17px; font-weight:400; opacity:.9">
Clasificación corto/largo + regresión especializada para SPARCS
</h2>

<hr style="border:none; border-top:1px solid rgba(255,255,255,.3); margin:0 0 18px 0"/>

<div style="display:flex; gap:40px; font-size:13px; opacity:.85">

<div>
<b>Autores:</b><br>
Daniel Vallejo Vecil<br>
Juliana Muñoz González
</div>

<div>
<b>Director:</b><br>
Julián F. Sánchez López
</div>

<div>
<b>Capa construida:</b><br>
Modelos
</div>

</div>

</div>

<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
1. Objetivo y diseño metodológico
</h2>

</div>

<div style="background:#F3E5F5;
border-left:6px solid #4A148C;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


<code>Length of Stay</code> concentra una proporción muy alta de casos en 1-2 días (17.6% con
exactamente 1 día), con una cola larga hacia estancias prolongadas. Mezclar ambos comportamientos en
un solo modelo de regresión diluye la señal de la cola larga (que es clínicamente la más relevante)
con el ruido de las estancias cortas.

<b>Diseño de dos etapas (hurdle model), siguiendo a Xu et al. (2022):</b>

<ol>
<li><b>Etapa 1 — Clasificación:</b> ¿la estancia será <b>corta</b> (≤2 días) o <b>larga</b>
(&gt;2 días)?</li>
<li><b>Etapa 2 — Regresión:</b> solo para los casos que la Etapa 1 clasifica como estancia larga, un
regresor predice el número exacto de días.</li>
<li><b>Predicción final combinada:</b> si la Etapa 1 predice "corta", se asigna la mediana de estancia
corta observada en entrenamiento; si predice "larga", se usa la salida de la Etapa 2.</li>
</ol>

<b>Importante — esto no cambia el objetivo de la tesis (OE3):</b> se sigue prediciendo la DEH para el
100% de los pacientes. Por eso se reporta el desempeño del sistema completo sobre el 100% del conjunto
de prueba, no solo el desempeño del regresor sobre su subconjunto — así el resultado responde a la
pregunta original ("¿qué tan bien predigo la estancia de cualquier paciente que llega?") y no a una
pregunta más fácil.

Se mantiene la simetría con los 3 algoritmos usados en el resto de la tesis: Regresión
(Logística/Lineal), Random Forest y XGBoost, ahora en sus versiones de clasificación y regresión.


</div>

<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
2. Configuración y carga
</h2>

</div>

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import TweedieRegressor, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"
GOLD_PATH = DATA_PATH / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)

TABLAS_PATH = PROJECT_ROOT / "tablas"
TABLAS_PATH.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NUM = "04"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# IMPORTANTE: el split ya se hizo en 03_FeatureEngineering (necesario para el Target Encoding
# sin fuga de información). Este notebook NO debe volver a hacer train_test_split.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")


X_train: (1672271, 189)   y_train: (1672271,)
X_test:  (418068, 189)   y_test:  (418068,)


<div style="background:#1976D2;
padding:15px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h3 style="margin:0;">
2.1 Nota de rendimiento
</h3>

</div>

<div style="background:#FFF5F5;
border-left:6px solid #C62828;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Con ~1.67 millones de filas y ~209 columnas, Random Forest y XGBoost pueden requerir varios minutos y
bastante memoria RAM. Los hiperparámetros de esta versión (<code>n_estimators=100</code>,
<code>max_depth</code> limitado, <code>tree_method="hist"</code> en XGBoost) están ajustados para ser
razonables a esta escala. Si tu equipo tiene poca RAM y el proceso se cuelga o tarda demasiado, baja
<code>n_estimators</code> a 50 o reduce <code>max_depth</code> aún más.


</div>

<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
3. Definir el umbral corto/largo
</h2>

</div>

In [16]:
UMBRAL_CORTO_LARGO = 3  # días: <= 2 es "corta", > 2 es "larga"

y_train_binaria = (y_train > UMBRAL_CORTO_LARGO).astype(int)  # 1 = larga, 0 = corta
y_test_binaria = (y_test > UMBRAL_CORTO_LARGO).astype(int)

print(f"Entrenamiento: {(y_train_binaria==0).sum():,} cortas ({(y_train_binaria==0).mean()*100:.1f}%) | "
      f"{(y_train_binaria==1).sum():,} largas ({(y_train_binaria==1).mean()*100:.1f}%)")
print(f"Prueba:        {(y_test_binaria==0).sum():,} cortas ({(y_test_binaria==0).mean()*100:.1f}%) | "
      f"{(y_test_binaria==1).sum():,} largas ({(y_test_binaria==1).mean()*100:.1f}%)")

# Valor de referencia para asignar cuando el sistema predice "estancia corta"
VALOR_CORTO = y_train[y_train <= UMBRAL_CORTO_LARGO].median()
print(f"\nValor asignado a las predicciones de 'estancia corta': {VALOR_CORTO} días (mediana real de ese grupo)")


Entrenamiento: 907,528 cortas (54.3%) | 764,743 largas (45.7%)
Prueba:        226,935 cortas (54.3%) | 191,133 largas (45.7%)

Valor asignado a las predicciones de 'estancia corta': 2.0 días (mediana real de ese grupo)


<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
4. Etapa 1 — Clasificación (corta vs. larga)
</h2>

</div>

In [17]:
clasificadores = {
    "Regresion Logistica": LogisticRegression(max_iter=1000, n_jobs=-1),
    "Random Forest Clasificador": RandomForestClassifier(
        n_estimators=100, max_depth=20, random_state=42, n_jobs=-1
    ),
    "XGBoost Clasificador": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist",
        random_state=42, n_jobs=-1, eval_metric="logloss"
    ),
}

modelos_etapa1 = {}
resultados_etapa1 = []

for nombre, modelo in clasificadores.items():
    print(f"Entrenando: {nombre}...")
    modelo.fit(X_train, y_train_binaria)
    pred = modelo.predict(X_test)

    metrics = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test_binaria, pred),
        "precision": precision_score(y_test_binaria, pred),
        "recall": recall_score(y_test_binaria, pred),
        "f1": f1_score(y_test_binaria, pred),
    }
    resultados_etapa1.append(metrics)
    modelos_etapa1[nombre] = modelo

    print(f"  Accuracy={metrics['accuracy']:.3f}  Precision={metrics['precision']:.3f}  "
          f"Recall={metrics['recall']:.3f}  F1={metrics['f1']:.3f}")

    joblib.dump(modelo, MODELOS_PATH / f"etapa1_{nombre.replace(' ', '_')}.joblib")

resultados_etapa1_df = pd.DataFrame(resultados_etapa1)
display(resultados_etapa1_df)
guardar_tabla(resultados_etapa1_df, "ResultadosEtapa1Clasificacion")


Entrenando: Regresion Logistica...


c:\Users\danie\miniconda3\envs\tesis\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  Accuracy=0.763  Precision=0.749  Recall=0.725  F1=0.737
Entrenando: Random Forest Clasificador...
  Accuracy=0.776  Precision=0.758  Recall=0.750  F1=0.754
Entrenando: XGBoost Clasificador...
  Accuracy=0.781  Precision=0.761  Recall=0.761  F1=0.761


,modelo,accuracy,precision,recall,f1
0,Regresion Logistica,0.763206,0.748991,0.725040,0.736821
1,Random Forest Clasificador,0.776477,0.758157,0.750477,0.754298
2,XGBoost Clasificador,0.781193,0.760687,0.760727,0.760707


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_ResultadosEtapa1Clasificacion.csv


<div style="background:#1976D2;
padding:15px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h3 style="margin:0;">
4.1 Matrices de confusión
</h3>

</div>

In [18]:
for nombre, modelo in modelos_etapa1.items():
    pred = modelo.predict(X_test)
    cm = confusion_matrix(y_test_binaria, pred)
    cm_df = pd.DataFrame(cm, index=["Real: Corta", "Real: Larga"], columns=["Pred: Corta", "Pred: Larga"])
    print(f"--- {nombre} ---")
    display(cm_df)
    guardar_tabla(cm_df, f"MatrizConfusion_{nombre.replace(' ', '_')}")
    print()


--- Regresion Logistica ---


,Pred: Corta,Pred: Larga
Real: Corta,180493,46442
Real: Larga,52554,138579


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_MatrizConfusion_Regresion_Logistica.csv

--- Random Forest Clasificador ---


,Pred: Corta,Pred: Larga
Real: Corta,181179,45756
Real: Larga,47692,143441


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_MatrizConfusion_Random_Forest_Clasificador.csv

--- XGBoost Clasificador ---


,Pred: Corta,Pred: Larga
Real: Corta,181192,45743
Real: Larga,45733,145400


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_MatrizConfusion_XGBoost_Clasificador.csv



<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
5. Etapa 2 — Regresión (solo estancia larga)
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Se entrena <b>únicamente</b> con los registros de entrenamiento donde la estancia real es larga
(&gt;2 días) — el regresor nunca ve estancias cortas, se especializa en la parte difícil del
problema.


</div>

In [19]:
mask_largo_train = y_train_binaria == 1
mask_largo_test = y_test_binaria == 1

X_train_largo = X_train[mask_largo_train]
y_train_largo = y_train[mask_largo_train]
X_test_largo = X_test[mask_largo_test]
y_test_largo = y_test[mask_largo_test]

print(f"Entrenamiento (solo largas): {X_train_largo.shape}")
print(f"Prueba (solo largas, verdadero positivo real): {X_test_largo.shape}")

regresores = {
    "Regresion Lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(
        n_estimators=500, max_depth=12, learning_rate=0.05, subsample=0.8,
        tree_method="hist", random_state=42, n_jobs=-1
    ),
}

modelos_etapa2 = {}
resultados_etapa2 = []

for nombre, modelo in regresores.items():
    print(f"Entrenando: {nombre}...")
    modelo.fit(X_train_largo, y_train_largo)
    pred = modelo.predict(X_test_largo)

    mae = mean_absolute_error(y_test_largo, pred)
    rmse = np.sqrt(mean_squared_error(y_test_largo, pred))
    r2 = r2_score(y_test_largo, pred)

    resultados_etapa2.append({"modelo": nombre, "MAE": mae, "RMSE": rmse, "R2": r2})
    modelos_etapa2[nombre] = modelo

    print(f"  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}")

    joblib.dump(modelo, MODELOS_PATH / f"etapa2_{nombre.replace(' ', '_')}.joblib")

resultados_etapa2_df = pd.DataFrame(resultados_etapa2)
display(resultados_etapa2_df)
guardar_tabla(resultados_etapa2_df, "ResultadosEtapa2Regresion")


Entrenamiento (solo largas): (764743, 189)
Prueba (solo largas, verdadero positivo real): (191133, 189)
Entrenando: Regresion Lineal...
  MAE=4.930  RMSE=8.796  R2=0.316
Entrenando: Random Forest...
  MAE=4.533  RMSE=8.451  R2=0.369
Entrenando: XGBoost...
  MAE=4.468  RMSE=8.320  R2=0.388


,modelo,MAE,RMSE,R2
0,Regresion Lineal,4.929575,8.796006,0.315967
1,Random Forest,4.532663,8.451274,0.368533
2,XGBoost,4.467778,8.319850,0.388020


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_ResultadosEtapa2Regresion.csv


<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Estas métricas se calculan sobre <code>X_test_largo</code>: la porción del conjunto de prueba donde
la estancia <b>real</b> ya es larga. Miden qué tan bien el regresor predice el número de días
<i>dado que</i> se sabe que el paciente va a tener estancia larga — es la pregunta más difícil y más
clínicamente relevante, pero <b>no es todavía</b> el resultado final del sistema (ver Sección 6).


</div>

<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
6. Sistema combinado: desempeño sobre el 100% del conjunto de prueba
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Aquí se responde la pregunta completa de la tesis. Para cada fila del conjunto de prueba completo
(sin filtrar por si es corta o larga):

<ol>
<li>El clasificador de la Etapa 1 predice si es corta o larga.</li>
<li>Si predice corta → predicción final = mediana de estancia corta (<code>VALOR_CORTO</code>).</li>
<li>Si predice larga → predicción final = salida del regresor de la Etapa 2 correspondiente.</li>
</ol>

Se arman 3 "sistemas" pareando cada clasificador con su regresor equivalente (Regresión, Random
Forest, XGBoost), y se evalúa cada sistema completo contra <code>y_test</code> real (el 100%, no un
subconjunto).


</div>

In [20]:
PARES_SISTEMA = [
    ("Regresion Logistica", "Regresion Lineal", "Sistema Regresion (Logistica + Lineal)"),
    ("Random Forest Clasificador", "Random Forest", "Sistema Random Forest"),
    ("XGBoost Clasificador", "XGBoost", "Sistema XGBoost"),
]

resultados_sistema = []

for nombre_clf, nombre_reg, nombre_sistema in PARES_SISTEMA:
    clf = modelos_etapa1[nombre_clf]
    reg = modelos_etapa2[nombre_reg]

    pred_clase = clf.predict(X_test)  # 0 = corta, 1 = larga

    pred_final = np.full(len(X_test), VALOR_CORTO, dtype=float)
    idx_largas_predichas = np.where(pred_clase == 1)[0]

    if len(idx_largas_predichas) > 0:
        pred_final[idx_largas_predichas] = reg.predict(X_test.iloc[idx_largas_predichas])

    mae = mean_absolute_error(y_test, pred_final)
    rmse = np.sqrt(mean_squared_error(y_test, pred_final))
    r2 = r2_score(y_test, pred_final)

    resultados_sistema.append({
        "sistema": nombre_sistema, "MAE": mae, "RMSE": rmse, "R2": r2,
        "pct_predichas_largas": (pred_clase == 1).mean() * 100,
    })

    print(f"{nombre_sistema}: MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}  "
          f"({(pred_clase==1).mean()*100:.1f}% del test predicho como estancia larga)")

resultados_sistema_df = pd.DataFrame(resultados_sistema)
display(resultados_sistema_df)
guardar_tabla(resultados_sistema_df, "ResultadosSistemaCombinado")


Sistema Regresion (Logistica + Lineal): MAE=3.523  RMSE=6.684  R2=0.352  (44.3% del test predicho como estancia larga)
Sistema Random Forest: MAE=3.260  RMSE=6.397  R2=0.406  (45.3% del test predicho como estancia larga)
Sistema XGBoost: MAE=3.201  RMSE=6.289  R2=0.426  (45.7% del test predicho como estancia larga)


,sistema,MAE,RMSE,R2,pct_predichas_largas
0,Sistema Regresion (Logistica + Lineal),3.523472,6.684005,0.351806,44.256198
1,Sistema Random Forest,3.259693,6.397406,0.406201,45.255078
2,Sistema XGBoost,3.200794,6.288684,0.426213,45.720553


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_ResultadosSistemaCombinado.csv


<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
7. Comparación: sistema de dos etapas vs. modelo único (referencia)
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Como referencia, se entrena también un XGBoost de una sola etapa (sin el hurdle model) sobre el 100%
del conjunto de entrenamiento, para poder cuantificar en la tesis cuánto aportó la arquitectura de dos
etapas frente al enfoque simple de un solo modelo.


</div>

In [21]:
modelo_unico = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
    tree_method="hist", random_state=42, n_jobs=-1
)
print("Entrenando modelo de referencia (una sola etapa, XGBoost)...")
modelo_unico.fit(X_train, y_train)
pred_unico = modelo_unico.predict(X_test)

mae_unico = mean_absolute_error(y_test, pred_unico)
rmse_unico = np.sqrt(mean_squared_error(y_test, pred_unico))
r2_unico = r2_score(y_test, pred_unico)

print(f"XGBoost de una sola etapa: MAE={mae_unico:.3f}  RMSE={rmse_unico:.3f}  R2={r2_unico:.3f}")

joblib.dump(modelo_unico, MODELOS_PATH / "referencia_xgboost_una_etapa.joblib")

comparacion_final = pd.concat([
    resultados_sistema_df[["sistema", "MAE", "RMSE", "R2"]].rename(columns={"sistema": "modelo"}),
    pd.DataFrame([{"modelo": "XGBoost (una sola etapa, referencia)",
                    "MAE": mae_unico, "RMSE": rmse_unico, "R2": r2_unico}])
], ignore_index=True)

display(comparacion_final)
guardar_tabla(comparacion_final, "ComparacionFinalHurdleVsUnico")


Entrenando modelo de referencia (una sola etapa, XGBoost)...
XGBoost de una sola etapa: MAE=2.993  RMSE=6.176  R2=0.447


,modelo,MAE,RMSE,R2
0,Sistema Regresion (Logistica + Lineal),3.523472,6.684005,0.351806
1,Sistema Random Forest,3.259693,6.397406,0.406201
2,Sistema XGBoost,3.200794,6.288684,0.426213
3,"XGBoost (una sola etapa, referencia)",2.993304,6.175792,0.446628


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\04_ComparacionFinalHurdleVsUnico.csv


<div style="background:#4A148C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
8. Conclusiones
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


La Tabla generada en la Sección 7 (<code>04_ComparacionFinalHurdleVsUnico.csv</code>) es la
comparación central de este notebook: contrasta los 3 sistemas de dos etapas contra un modelo único
de referencia, todos evaluados sobre el 100% del conjunto de prueba con las mismas métricas (MAE,
RMSE, R²). Esta tabla es la que reemplaza/complementa la Tabla 4.1 original de la tesis.

Adicionalmente, la Sección 4 documenta qué tan bien se distingue estancia corta de larga (Etapa 1), y
la Sección 5 qué tan bien se predice el número exacto de días dado que la estancia es larga (Etapa
2) — ambas piezas del argumento metodológico completo.


</div>